# Personal Clip Studio - Kaggle GPU Server

Before running, add these Kaggle secrets under **Add-ons > Secrets**: `GOOGLE_API_KEY`, `NGROK_AUTHTOKEN`, and `CLIP_STUDIO_TOKEN`.

This notebook starts your private clipping API. Keep the final cell running while you use Clip Studio.

In [ ]:
# Replace this with YOUR fork after pushing the local commit.
REPOSITORY_URL = 'https://github.com/YOUR-USERNAME/YOUR-REPOSITORY.git'

if 'YOUR-USERNAME' in REPOSITORY_URL:
    raise ValueError('Set REPOSITORY_URL to your GitHub fork first.')

!git clone $REPOSITORY_URL clip-studio
%cd clip-studio
!pip install -q -r requirements.txt pyngrok nest-asyncio aiofiles
!apt-get -qq update && apt-get -qq install -y ffmpeg


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def required_secret(name):
    value = secrets.get_secret(name)
    if not value:
        raise ValueError(f'Missing Kaggle secret: {name}')
    return value

os.environ['GOOGLE_API_KEY'] = required_secret('GOOGLE_API_KEY')
os.environ['CLIP_STUDIO_TOKEN'] = required_secret('CLIP_STUDIO_TOKEN')
os.environ['CLIP_STUDIO_ORIGINS'] = 'http://127.0.0.1:5173,https://zavtsar2.github.io'
NGROK_AUTHTOKEN = required_secret('NGROK_AUTHTOKEN')

print('Gemini key loaded')
print('Studio token loaded')
print('Allowed sites: local Clip Studio and https://zavtsar2.github.io')


In [ ]:
import subprocess
import time
import requests
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)
server = subprocess.Popen(['uvicorn', 'web.api.app:app', '--host', '0.0.0.0', '--port', '8000'])

for _ in range(30):
    try:
        response = requests.get('http://127.0.0.1:8000/api/health', headers={'Authorization': f"Bearer {os.environ['CLIP_STUDIO_TOKEN']}"}, timeout=2)
        if response.ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('The API did not start. Review the notebook output above.')

tunnel = ngrok.connect(8000)
print('\nClip Studio API is ready')
print(f'Tunnel URL: {tunnel.public_url}')
print('Open http://127.0.0.1:5173, select Connect notebook, and paste:')
print('  1. Tunnel URL above')
print('  2. Your CLIP_STUDIO_TOKEN Kaggle secret')
print('Keep this Kaggle session running while clipping.')
